# Figure 1 — inference-time mitigation on memorized CelebA-HQ seeds

Renders `fig1.{pdf,png}`: 5 rows (main seeds 22, 20, 21, 11, 15) x 3 blocks
(`Unguided`/`FR`/`Bayesian FR`), generated image beside its nearest training image. The
nearest image is the actual CelebA-HQ file (un-mirrored as needed), not a VAE round-trip of
the retrieved latent.

## Cost

32-step EDM sampler in 4x32x32 latent space, a few seconds/run. 15 runs plus decoding finish
in a few minutes. Guidance settings match `experiments/celeba_hq/02_fr_guidance.ipynb`.
Cache is per-run, so tuning `DIRICHLET_ALPHA` only re-runs the Bayesian arm.

In [1]:
import os, sys, time, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import UNet2DModel, AutoencoderKL

REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "workshop_figures"))

from guidance import driver, basic_fr, bayesian_fr
import wsstyle

torch.backends.cuda.enable_cudnn_sdp(False)   # see train_celeba_hq_edm.py's docstring

mpl_fs = 19
wsstyle.use_style(font_size=mpl_fs)

In [ ]:
EXP = os.path.join(REPO, "experiments", "celeba_hq")
CACHE = os.path.join(REPO, "workshop_figures", "cache", "fig1_runs.pt")

device = os.environ.get("DEVICE") or (
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu"))

config = {
    "device": device,
    "ckpt_path": os.path.join(EXP, "checkpoints", "celeba_hq_edm_finetuned.pt"),
    "vae_dir": os.path.join(EXP, "pretrained", "sd-vae-ft-mse"),
    "latents_path": os.path.join(EXP, "checkpoints", "celeba_hq_latents.pt"),
    "finetune_idx_path": os.path.join(EXP, "checkpoints", "celeba_hq_finetune_train_idx.npy"),
    "train_images_dir": os.path.join(REPO, "data", "celeba_hq", "images"),
    "n_steps": 32,
    "sigma_min": 0.002, "sigma_max": 80.0, "rho": 7.0,
    "mem_ratio_threshold": 1 / 3,
    "k_neighbors": 15,
    "ell": 1,
}

# matches experiments/celeba_hq/02_fr_guidance.ipynb
DEFAULTS = {"window_lo": 0.35, "window_hi": 1, "eta": 0.2, "q_target": 0.8}
PER_SEED = {}   # per-seed overrides; empty = use DEFAULTS

# alpha -> infinity recovers uniform prior (plain FR); LARGER alpha = weaker bootstrap
DIRICHLET_ALPHA, B_BOOTSTRAP = 0.2, 256

SEEDS = [22, 20, 21, 11, 15]   # the 5 seeds reported in the manuscript

METHODS = ["unguided", "basic", "bayesian"]
METHOD_LABELS = {"unguided": "Unguided", "basic": "FR", "bayesian": "Bayesian FR"}

D_LATENT = 4 * 32 * 32
N_STEPS = config["n_steps"]


def settings_for(seed):
    return {**DEFAULTS, **PER_SEED.get(seed, {})}


def run_key(seed, method):
    """Cache key: dirichlet_alpha/B only appear in the bayesian key, so retuning them
    leaves unguided/FR entries valid."""
    if method == "unguided":
        return f"unguided|seed={seed}"
    st = settings_for(seed)
    common = (f"seed={seed}|lo={st['window_lo']}|hi={st['window_hi']}"
              f"|eta={st['eta']}|qt={st['q_target']}|k={config['k_neighbors']}")
    if method == "basic":
        return f"basic|{common}"
    return f"bayesian|{common}|alpha={DIRICHLET_ALPHA}|B={B_BOOTSTRAP}"


def pack(x):
    return x.reshape(x.shape[0], -1)


def unpack(z):
    return z.reshape(z.shape[0], 4, 32, 32)


print(f"device={device}")
print(f"seeds           {SEEDS}")
print(f"settings        {DEFAULTS}")
print(f"Bayesian FR     dirichlet_alpha={DIRICHLET_ALPHA}, B={B_BOOTSTRAP}")

## Load the finetuned model and the finetune-subset latents as the k-NN index

In [3]:
ckpt = torch.load(config["ckpt_path"], map_location="cpu")
sigma_data = ckpt["config"]["sigma_data"]
ema_state = ckpt["model_ema"]
del ckpt
gc.collect()

net = UNet2DModel(
    sample_size=32, in_channels=4, out_channels=4, layers_per_block=2,
    block_out_channels=(128, 256, 384, 384),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
)
net.load_state_dict(ema_state)
del ema_state
net = net.to(device).eval()
for p in net.parameters():
    p.requires_grad_(False)
print(f"loaded model, sigma_data={sigma_data:.4f}")

latents_blob = torch.load(config["latents_path"], map_location="cpu")
all_latents = latents_blob["latents"].float()
ft_latent_idx = np.load(config["finetune_idx_path"])
X_train = all_latents[ft_latent_idx].to(device)
X_train_flat = pack(X_train)
train_index = basic_fr.build_flat_index(X_train_flat)
del latents_blob, all_latents
gc.collect()
print(f"finetune subset: {len(ft_latent_idx)} latents, X_train_flat {tuple(X_train_flat.shape)}")

train_files = sorted(os.listdir(config["train_images_dir"]))
n_train_images = len(train_files)


def ft_idx_to_file(latent_idx):
    """finetune-subset row -> (filename, mirrored?). Latent pool is [unflipped 0..n-1,
    flipped n..2n-1] (see precompute_latents.py)."""
    pool_idx = int(ft_latent_idx[latent_idx])
    img_idx = pool_idx % n_train_images
    mirrored = pool_idx >= n_train_images
    return train_files[img_idx], mirrored

loaded model, sigma_data=0.9241
finetune subset: 200 latents, X_train_flat (200, 4096)


## Deterministic EDM sampler, guidance energies, and one run

In [4]:
def edm_precond(x, sigma):
    sigma = sigma.view(-1, 1, 1, 1)
    c_skip = sigma_data ** 2 / (sigma ** 2 + sigma_data ** 2)
    c_out = sigma * sigma_data / (sigma ** 2 + sigma_data ** 2).sqrt()
    c_in = 1.0 / (sigma ** 2 + sigma_data ** 2).sqrt()
    c_noise = 0.25 * sigma.log().flatten()
    F_x = net(c_in * x, c_noise).sample
    return c_skip * x + c_out * F_x


def make_sigmas(n_steps):
    step_idx = torch.arange(n_steps, dtype=torch.float64)
    sigmas = (config["sigma_max"] ** (1 / config["rho"]) + step_idx / (n_steps - 1) *
              (config["sigma_min"] ** (1 / config["rho"])
               - config["sigma_max"] ** (1 / config["rho"]))) ** config["rho"]
    return torch.cat([sigmas, torch.zeros(1)]).float().to(device)


def make_wiring(sigmas):
    def base_step(z_t, step_idx):
        s_cur, s_next = sigmas[step_idx - 1], sigmas[step_idx]
        x_t = unpack(z_t)
        d_cur = (x_t - edm_precond(x_t, s_cur.expand(1))) / s_cur
        x_next = x_t + (s_next - s_cur) * d_cur
        if s_next > 0:
            d_next = (x_next - edm_precond(x_next, s_next.expand(1))) / s_next
            x_next = x_t + (s_next - s_cur) * 0.5 * (d_cur + d_next)
        return pack(x_next)

    def y_fn(z_t):
        return pack(edm_precond(unpack(z_t), y_fn.s.expand(1)))

    def base_step_with_s(z_t, step_idx):
        y_fn.s = sigmas[step_idx - 1]
        return base_step(z_t, step_idx)

    def sigma2_fn():
        return (y_fn.s ** 2).expand(1)

    return y_fn, base_step_with_s, sigma2_fn


def z_init_from(seed, sigmas):
    generator = torch.Generator(device=device if device != "mps" else "cpu").manual_seed(seed)
    return pack(torch.randn(1, 4, 32, 32, generator=generator).to(device) * sigmas[0])


def make_energy_fn(method, sigma2_fn, q_target):
    """basic/bayesian share retrieval and temperature, differ only in how E/beta becomes
    a posterior."""
    def energy_fn(y):
        with torch.no_grad():
            neighbors, sq_d = basic_fr.ann_query(train_index, y, config["k_neighbors"])
        bf = basic_fr.beta_soft_for(sq_d, q_target)
        if method == "basic":
            I, _, _ = basic_fr.fisher_rao_energy(y, neighbors, sigma2_fn(), beta_soft=bf)
        elif method == "bayesian":
            I, _, _ = bayesian_fr.fisher_rao_energy_bb(
                y, neighbors, sigma2_fn(), B=B_BOOTSTRAP,
                dirichlet_alpha=DIRICHLET_ALPHA, beta_soft=bf)
        else:
            raise ValueError(method)
        return I
    return energy_fn


def eval_memorization(z_single):
    d = torch.cdist(pack(z_single), X_train_flat)[0]
    top2 = torch.topk(d, 2, largest=False)
    d1, d2 = top2.values[0].item(), top2.values[1].item()
    ratio = d1 / d2
    return {"d1": d1, "d2": d2, "ratio": ratio, "nn_idx": int(top2.indices[0]),
            "is_memorized": ratio < config["mem_ratio_threshold"]}


def run_condition(seed, method):
    st = settings_for(seed)
    sigmas = make_sigmas(N_STEPS)
    y_fn, base_step_with_s, sigma2_fn = make_wiring(sigmas)
    z_init = z_init_from(seed, sigmas)

    if method == "unguided":
        z = z_init
        with torch.no_grad():
            for step_idx in range(1, N_STEPS + 1):
                z = base_step_with_s(z, step_idx)
    else:
        guidance_fn = driver.make_autograd_guidance_fn(
            y_fn, make_energy_fn(method, sigma2_fn, st["q_target"]), target_range=None)
        z, _ = driver.guided_reverse_loop(
            z_init, N_STEPS, base_step_with_s, guidance_fn,
            progress_lo=st["window_lo"], progress_hi=st["window_hi"],
            ell=config["ell"], trust_region=st["eta"])

    z = z.detach()
    stats = eval_memorization(unpack(z))
    return z.cpu(), stats

## Run every condition that is not already cached

15 runs (5 seeds x 3 arms).

In [ ]:
FORCE = []          # e.g. ["bayesian"] to re-run that arm even when cached

blob = torch.load(CACHE, weights_only=False) if os.path.exists(CACHE) else {}
store = blob.get("runs", {})
images = blob.get("images", {})
if store:
    print(f"cache holds {len(store)} runs, {len(images)} decoded")

planned = [(s, m) for s in SEEDS for m in METHODS]
todo = [(s, m) for s, m in planned if run_key(s, m) not in store or m in FORCE]
print(f"{len(planned) - len(todo)} reused, {len(todo)} to run")
print()

for seed, method in todo:
    t0 = time.time()
    z, stats = run_condition(seed, method)
    store[run_key(seed, method)] = {"seed": seed, "method": method, "z": z, "stats": stats,
                                     "settings": settings_for(seed),
                                     "alpha": DIRICHLET_ALPHA if method == "bayesian" else None}
    fname, mirrored = ft_idx_to_file(stats["nn_idx"])
    print(f"seed={seed:>3} {method:<9} ratio={stats['ratio']:.3f} "
          f"memorized={str(stats['is_memorized']):<5} nn={stats['nn_idx']:<5} "
          f"{fname}{' (mirrored)' if mirrored else ''} ({time.time() - t0:.1f}s)")

## Decode to pixels

VAE loaded lazily, only if something needs decoding. Decoded results stored as `uint8` to
keep the cache small.

In [ ]:
DISPLAY_RES = 256


def to_u8(img_pm1):
    """[3,H,W] in [-1,1] -> uint8 [3,H,W]."""
    return (((img_pm1 + 1) / 2).clamp(0, 1) * 255).round().to(torch.uint8)


need = [k for k in (run_key(s, m) for s in SEEDS for m in METHODS) if k not in images]
print(f"{len(need)} tiles to decode")

if need:
    vae = AutoencoderKL.from_pretrained(config["vae_dir"]).to(device).eval()

    def decode(z_single):
        with torch.no_grad():
            img = vae.decode(unpack(z_single.to(device)) / vae.config.scaling_factor).sample
        return img.clamp(-1, 1)[0].cpu()

    def load_train_image(nn_idx):
        fname, mirrored = ft_idx_to_file(nn_idx)
        img = Image.open(os.path.join(config["train_images_dir"], fname)).convert("RGB")
        if mirrored:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = img.resize((DISPLAY_RES, DISPLAY_RES), Image.LANCZOS)
        arr = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)
        return arr.to(torch.uint8), fname, mirrored

    for k in need:
        rec = store[k]
        nn_u8, fname, mirrored = load_train_image(rec["stats"]["nn_idx"])
        images[k] = {"gen": to_u8(decode(rec["z"])), "nn": nn_u8,
                     "nn_file": fname, "mirrored": mirrored}
    del vae
    gc.collect()
    print("decoded")

torch.save({"runs": store, "images": images}, CACHE)
print("saved ->", CACHE)

print()
print(f"caption numbers  (Bayesian FR: dirichlet_alpha={DIRICHLET_ALPHA}, B={B_BOOTSTRAP}, "
      f"eta={DEFAULTS['eta']}, q_target={DEFAULTS['q_target']}, "
      f"window=({DEFAULTS['window_lo']}, {DEFAULTS['window_hi']}))")
for seed in SEEDS:
    bits = "  ".join(
        f"{m}: ratio={store[run_key(seed, m)]['stats']['ratio']:.3f} "
        f"mem={store[run_key(seed, m)]['stats']['is_memorized']}" for m in METHODS)
    print(f"  seed {seed:>3}: {bits}")

## Render

Light rule grid, column titles, row labels on the left, no hyperparameters in the figure.
Nearest-training tiles get a lighter frame.

In [ ]:
def draw_figure(seeds, name):
    BLOCKS = [(METHOD_LABELS[m], m) for m in METHODS]
    COL_TITLES = ["Generated", "Nearest train"]

    fig = plt.figure(figsize=(15.0, 13.0))
    axes, blocks, block_rects = wsstyle.place_grid(
        fig, n_rows=len(seeds), block_sizes=[2] * len(BLOCKS),
        left=0.058, right=0.012, top=0.880, bottom=0.030,
        block_gap=0.050, panel_gap=0.012, row_gap=0.020)

    for i, seed in enumerate(seeds):
        for b, (_, method) in enumerate(BLOCKS):
            rec = images[run_key(seed, method)]
            for j, key in enumerate(["gen", "nn"]):
                ax = blocks[b][i][j]
                ax.imshow(rec[key].permute(1, 2, 0).numpy(), interpolation="nearest")
                wsstyle.strip_axes(ax)
                for sp in ax.spines.values():
                    sp.set_visible(True)
                    sp.set_linewidth(1.0)
                    sp.set_edgecolor("#7A7A7A" if j == 0 else "#D0D0D0")

    for i, seed in enumerate(seeds):
        axes[i][0].set_ylabel(f"seed {seed}", labelpad=14)

    for b in range(len(BLOCKS)):
        for ax, t in zip(blocks[b][0], COL_TITLES):
            ax.set_title(t, fontsize=mpl_fs - 3, pad=8)

    rule_ys = []
    for (label, _), block, rect in zip(BLOCKS, blocks, block_rects):
        flat = wsstyle.flatten(block)
        y = wsstyle.block_header(fig, flat, label, x_extent=rect, dy=0.052)
        wsstyle.header_rule(fig, rect, y - 0.010)
        rule_ys.append(y - 0.010)

    y_top = max(rule_ys)
    y_bottom = min(ax.get_position().y0 for ax in wsstyle.flatten(axes)) - 0.012
    for x in wsstyle.gutter_positions(block_rects):
        wsstyle.separator(fig, x, y_bottom, y_top, color="#C4C4C4")

    return wsstyle.save(fig, name)


print(draw_figure(SEEDS, "fig1"))